# TTS Integration

Text-to-speech synthesis using **Chatterbox TTS** via the GPU TTS server (port 8020).
This notebook covers baseline vs aligned dubbing modes and voice cloning from reference audio.

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

# Load .env (LOGFIRE_TOKEN, HF_TOKEN, etc.)
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from foreign_whispers.client import FWClient, BASELINE, ALIGNED

fw = FWClient("http://localhost:8080")
fw.healthz()

# Optional: Logfire tracing (no-op shim if unavailable)
try:
    import logfire
    logfire.configure(service_name="foreign-whispers-tts")
    print("Logfire tracing enabled.")
except Exception:
    class _NoopSpan:
        def __enter__(self): return self
        def __exit__(self, *a): pass
    class _noop:
        @staticmethod
        def span(name, **kw): return _NoopSpan()
        @staticmethod
        def info(*a, **kw): pass
    logfire = _noop()
    print("Logfire not configured — using no-op shim.")

## Baseline TTS (No Alignment)

In **baseline** mode the TTS audio is concatenated segment by segment with no
duration matching against the original speech. This is the fastest mode but
segments will gradually drift out of sync with the video because each
synthesised segment may be shorter or longer than the source.

In [ ]:
video_id = "GYQ5yGV_-Oc"  # change to your video

with logfire.span("tts.baseline", video_id=video_id):
    baseline_result = fw.tts(video_id, config=BASELINE, alignment=False)

print(f"Audio path: {baseline_result['audio_path']}")
print(f"Config:     {baseline_result['config']}")

## Aligned TTS

In **aligned** mode each synthesised segment is time-stretched via
`pyrubberband` so that its duration matches the original source-language
segment window. This is slower but keeps the dubbed audio synchronised
with the video.

In [ ]:
with logfire.span("tts.aligned", video_id=video_id):
    aligned_result = fw.tts(video_id, config=ALIGNED, alignment=True)

print(f"Audio path: {aligned_result['audio_path']}")
print(f"Config:     {aligned_result['config']}")

## Compare Audio Durations

Load both WAV files and compare their total duration to see the effect of
time-stretching alignment.

In [ ]:
tts_root = PROJECT_ROOT / "pipeline_data" / "api" / "tts_audio" / "chatterbox"

baseline_dir = tts_root / BASELINE
aligned_dir = tts_root / ALIGNED


def wav_duration(wav_path: Path) -> float:
    """Return duration in seconds. Uses scipy if available, else file-size estimate."""
    try:
        from scipy.io import wavfile

        sr, data = wavfile.read(wav_path)
        return len(data) / sr
    except ImportError:
        # rough estimate: assume 16-bit mono 22050 Hz
        size = wav_path.stat().st_size - 44  # subtract WAV header
        return size / (22050 * 2)


for label, d in [("Baseline", baseline_dir), ("Aligned", aligned_dir)]:
    wavs = sorted(d.glob(f"{video_id}*.wav")) if d.exists() else []
    if wavs:
        dur = wav_duration(wavs[0])
        print(f"{label:10s}  {wavs[0].name}  duration={dur:.2f}s")
    else:
        print(f"{label:10s}  no WAV found in {d}")

## Speaker Reference Voices

Chatterbox supports voice cloning from short reference WAV clips. Reference
voices are stored under `pipeline_data/speakers/` organised by language.

In [ ]:
speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"

if speakers_dir.exists():
    for lang_dir in sorted(speakers_dir.iterdir()):
        if lang_dir.is_dir():
            wavs = sorted(lang_dir.glob("*.wav"))
            print(f"{lang_dir.name}/  ({len(wavs)} WAV files)")
            for w in wavs:
                size_kb = w.stat().st_size / 1024
                print(f"  {w.name}  ({size_kb:.1f} KB)")
else:
    print(f"Speakers directory not found: {speakers_dir}")

---

## Voice Cloning Integration (Implemented)

The Chatterbox container supports voice cloning via `/v1/audio/speech/upload`, and the Foreign Whispers pipeline now passes a speaker reference WAV whenever diarization has labeled a segment and a matching reference exists.

Current behavior:

- explicit `speaker_wav` query parameter wins when provided
- otherwise, diarized `speaker` labels resolve through `foreign_whispers.voice_resolution.resolve_speaker_wav()`
- references are looked up under `pipeline_data/speakers/{language}/{speaker}.wav`
- missing or invalid references fall back to the default Chatterbox endpoint instead of failing the pipeline
- speaker reference pitch is estimated to record `speaker_gender`, and fallback Flite voices choose male/female voice pools from that gender
- cached diarized TTS audio is considered stale if voice cloning is requested but the sidecar has no `speaker_wav`


### 1 - Inspect the Existing Chatterbox Client

The client already routes to the upload endpoint when `speaker_wav` is present. This cell prints the relevant implementation sections for quick review.


In [ ]:
# Read the relevant sections of tts.py
tts_path = PROJECT_ROOT / "tts.py"
lines = tts_path.read_text().splitlines()

print("=== Chatterbox Configuration (lines 15-18) ===")
for line in lines[14:18]:
    print(f"  {line}")

print("\n=== ChatterboxClient constructor (lines 34-44) ===")
for line in lines[33:44]:
    print(f"  {line}")

print("\n=== tts_to_file method (lines 46-75) ===")
for line in lines[45:75]:
    print(f"  {line}")

print("\n--- Key observation ---")
print("ChatterboxClient.tts_to_file() already accepts **kwargs including 'speaker_wav'.")
print("The Chatterbox container receives a voice_file upload via /v1/audio/speech/upload.")
print("But nothing in the API layer passes this parameter through.")

In [ ]:
# Explore what reference voices are available
speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"

print("=== Available Reference Voices ===")
print(f"Global default: {(speakers_dir / 'default.wav').exists()}")
print()
for lang_dir in sorted(speakers_dir.iterdir()):
    if lang_dir.is_dir():
        wavs = sorted(lang_dir.glob("*.wav"))
        if wavs:
            print(f"{lang_dir.name}/")
            for w in wavs:
                size_kb = w.stat().st_size / 1024
                print(f"  {w.name}  ({size_kb:.1f} KB)")
        else:
            print(f"{lang_dir.name}/  (empty — needs reference WAVs)")

---

### 2 - Voice Resolution Function (Implemented)

`foreign_whispers.voice_resolution.resolve_speaker_wav()` resolves a relative WAV path in this order:

1. `speakers/{lang}/{speaker_id}.wav`
2. `speakers/{lang}/default.wav`
3. `speakers/default.wav`

It also converts matching reference media files (`.mp4`, `.m4a`, `.mp3`, etc.) into 16 kHz mono WAV when needed.


#### 2.1 - Validate the voice resolver

Run these tests as a quick notebook-level smoke test. They should pass against the implemented `resolve_speaker_wav` helper.


In [ ]:
# These tests define the contract for resolve_speaker_wav.
# DO NOT modify the tests — make your implementation pass them.

import tempfile
import os

def run_voice_tests():
    try:
        from foreign_whispers.voice_resolution import resolve_speaker_wav
    except (ImportError, ModuleNotFoundError):
        print("✗ foreign_whispers.voice_resolution not found — create the file first (Task 2.2)")
        return False

    passed, failed = 0, 0

    # Create a temporary speakers directory structure for testing
    with tempfile.TemporaryDirectory() as tmpdir:
        speakers = Path(tmpdir)

        # Create test structure:
        #   speakers/default.wav
        #   speakers/es/default.wav
        #   speakers/es/SPEAKER_00.wav
        #   speakers/fr/  (empty — no WAVs)
        (speakers / "default.wav").write_bytes(b"RIFF" + b"\x00" * 40)
        (speakers / "es").mkdir()
        (speakers / "es" / "default.wav").write_bytes(b"RIFF" + b"\x00" * 40)
        (speakers / "es" / "SPEAKER_00.wav").write_bytes(b"RIFF" + b"\x00" * 40)
        (speakers / "fr").mkdir()

        # Test 1: Speaker-specific WAV exists
        try:
            result = resolve_speaker_wav(speakers, "es", "SPEAKER_00")
            assert result == "es/SPEAKER_00.wav", f"Expected 'es/SPEAKER_00.wav', got '{result}'"
            print("✓ Test 1 passed: speaker-specific WAV")
            passed += 1
        except Exception as e:
            print(f"✗ Test 1 FAILED: {e}")
            failed += 1

        # Test 2: No speaker-specific WAV, falls back to language default
        try:
            result = resolve_speaker_wav(speakers, "es", "SPEAKER_01")
            assert result == "es/default.wav", f"Expected 'es/default.wav', got '{result}'"
            print("✓ Test 2 passed: language default fallback")
            passed += 1
        except Exception as e:
            print(f"✗ Test 2 FAILED: {e}")
            failed += 1

        # Test 3: No language dir WAVs, falls back to global default
        try:
            result = resolve_speaker_wav(speakers, "fr", "SPEAKER_00")
            assert result == "default.wav", f"Expected 'default.wav', got '{result}'"
            print("✓ Test 3 passed: global default fallback")
            passed += 1
        except Exception as e:
            print(f"✗ Test 3 FAILED: {e}")
            failed += 1

        # Test 4: No speaker_id provided, uses language default
        try:
            result = resolve_speaker_wav(speakers, "es")
            assert result == "es/default.wav", f"Expected 'es/default.wav', got '{result}'"
            print("✓ Test 4 passed: no speaker_id uses language default")
            passed += 1
        except Exception as e:
            print(f"✗ Test 4 FAILED: {e}")
            failed += 1

        # Test 5: Unknown language, falls back to global default
        try:
            result = resolve_speaker_wav(speakers, "xx")
            assert result == "default.wav", f"Expected 'default.wav', got '{result}'"
            print("✓ Test 5 passed: unknown language fallback")
            passed += 1
        except Exception as e:
            print(f"✗ Test 5 FAILED: {e}")
            failed += 1

    print(f"\n{'='*40}")
    print(f"Results: {passed} passed, {failed} failed")
    return failed == 0

# Run — expected to FAIL at this point
run_voice_tests()

#### 2.2 - Review `resolve_speaker_wav`

The helper is already implemented in `foreign_whispers/voice_resolution.py`. Use this cell to inspect the source instead of creating a stub.


In [ ]:
import inspect
from foreign_whispers.voice_resolution import resolve_speaker_wav

print(inspect.getsource(resolve_speaker_wav))


#### 2.3 — Re-run the tests

After implementing `resolve_speaker_wav`, re-run the tests. All 5 should pass.

In [ ]:
# Reload and re-run (only works after you've implemented the function)
try:
    import importlib
    import foreign_whispers.voice_resolution
    importlib.reload(foreign_whispers.voice_resolution)
    if run_voice_tests():
        print("\nAll tests passed!")
    else:
        print("\nSome tests failed — fix your implementation before continuing.")
except (ImportError, ModuleNotFoundError):
    print("Skipped — create foreign_whispers/voice_resolution.py first (Task 2.2)")

#### 2.4 - Status

The voice resolution helper is implemented and covered by tests in the repo. The remaining notebook cells validate API and per-speaker behavior against the current pipeline.

---

### 3 - `speaker_wav` Parameter in the TTS API (Implemented)

`POST /api/tts/{video_id}` accepts `speaker_wav` for explicit voice selection. The service forwards it to `text_file_to_speech()`, and Chatterbox uploads that reference WAV for synthesis.


#### 3.1 — Add `speakers_dir` to Settings

Open `api/src/core/config.py` and add this property:

```python
@property
def speakers_dir(self) -> Path:
    return self.base_dir / "pipeline_data" / "speakers"
```

#### 3.2 — Add `speaker_wav` to the TTS endpoint

Open `api/src/routers/tts.py`. Add a new query parameter and pass it through:

```python
from foreign_whispers.voice_resolution import resolve_speaker_wav

@router.post("/tts/{video_id}")
async def tts_endpoint(
    video_id: str,
    request: Request,
    config: str = Query(..., pattern=r"^c-[0-9a-f]{7}$"),
    alignment: bool = Query(False),
    speaker_wav: str = Query(None, description="Reference voice WAV path (e.g. 'es/default.wav')"),
):
```

Inside the endpoint, if `speaker_wav` is not provided, resolve it automatically:

```python
if speaker_wav is None:
    speaker_wav = resolve_speaker_wav(settings.speakers_dir, "es")
```

#### 3.3 — Pass `speaker_wav` through the service

Open `api/src/services/tts_service.py`. Modify `text_file_to_speech` to accept and forward `speaker_wav`:

```python
def text_file_to_speech(self, source_path: str, output_path: str,
                        *, alignment: bool | None = None,
                        speaker_wav: str | None = None) -> None:
    tts_text_file_to_speech(source_path, output_path, self.tts_engine,
                            alignment=alignment, speaker_wav=speaker_wav)
```

Then in `tts.py`, the `text_file_to_speech` function must pass `speaker_wav` as a kwarg to `ChatterboxClient.tts_to_file()`. Find where `tts_to_file` is called and add:

```python
client.tts_to_file(text, wav_path, speaker_wav=speaker_wav)
```

#### 3.4 — Test manually

In [ ]:
# Rebuild and restart the API (run manually after implementing Task 3)
# !cd {PROJECT_ROOT} && docker compose --profile nvidia build api
# !cd {PROJECT_ROOT} && docker compose --profile nvidia up -d api
print("Uncomment the lines above and run this cell after implementing Task 3.")

In [ ]:
import requests

API_BASE = "http://localhost:8080"

# Test TTS with explicit speaker_wav (only works after Task 3 is implemented)
try:
    resp = requests.post(
        f"{API_BASE}/api/tts/{video_id}",
        params={"config": BASELINE, "alignment": "false", "speaker_wav": "es/default.wav"},
        timeout=10,
    )
    print(f"Status: {resp.status_code}")
    if resp.ok:
        print(resp.json())
    else:
        print(f"Error: {resp.text}")
except requests.ConnectionError:
    print("API not reachable — rebuild and restart after implementing Task 3.")

#### 3.5 - Status

The API/service wiring for `speaker_wav` is implemented and tested in `tests/test_tts_router.py` and `tests/test_tts_alignment_wire.py`.

---

### 4 - Per-Speaker Voice Assignment (Implemented)

When translated segments contain `speaker` labels from diarization, TTS now resolves a reference WAV per speaker by default during voice cloning.


#### 4.1 - Current Voice Assignment Strategy

- **Explicit override:** if `speaker_wav` is passed to the endpoint, every segment uses that reference.
- **Diarized voices:** otherwise, each segment's `speaker` field maps to `pipeline_data/speakers/{language}/{speaker}.wav`.
- **Gender matching:** `_speaker_gender()` estimates pitch from the reference WAV and records `male` or `female`; fallback Flite synthesis uses the matching gender voice pool.
- **Fallbacks:** if a reference WAV is missing, TTS still produces audio using the default endpoint plus speaker coloring metadata.
- **Cache safety:** cached audio without `speaker_wav` is regenerated when voice cloning is requested.


#### 4.3 — Test with a multi-speaker video

After implementing per-speaker voice assignment:

1. Run the diarization integration first to label segments with speaker IDs
2. Then run TTS — each speaker should use a different reference voice
3. Listen to the output to verify voice switching works

In [ ]:
# Verify voice mapping (after diarization has run)
import json

trans_dir = PROJECT_ROOT / "pipeline_data" / "api" / "translations" / "argos"
trans_files = list(trans_dir.glob("*.json"))

if trans_files:
    translated = json.loads(trans_files[0].read_text())
    segments = translated.get("segments", [])

    # Check which segments have speaker labels
    speakers = set()
    labeled = 0
    for seg in segments:
        spk = seg.get("speaker")
        if spk:
            speakers.add(spk)
            labeled += 1

    print(f"Total segments: {len(segments)}")
    print(f"With speaker labels: {labeled}")
    print(f"Unique speakers: {speakers or '(none — run diarization first)'}")

    if speakers:
        try:
            from foreign_whispers.voice_resolution import resolve_speaker_wav
            speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"
            print("\nVoice mapping:")
            for spk in sorted(speakers):
                voice = resolve_speaker_wav(speakers_dir, "es", spk)
                print(f"  {spk} → {voice}")
        except (ImportError, ModuleNotFoundError):
            print("\nSkipped voice mapping — implement resolve_speaker_wav first (Task 2)")
else:
    print("No translated transcripts found — run the pipeline first.")

#### 4.4 - Validation Checklist

Run the repo tests after changing voice behavior:

```bash
.venv\Scripts\python.exe -m pytest tests/test_tts_alignment_wire.py tests/test_tts_router.py
```

Expected behavior for a diarized, multi-speaker clip:

1. Run diarization to create translated segment `speaker` labels and `pipeline_data/speakers/es/<video>__SPEAKER_XX.wav` references.
2. Run TTS with `voice_cloning=true`.
3. Confirm `<audio>.align.json` records `speaker`, `speaker_wav`, `speaker_voice`, and `speaker_gender` per speaker segment.
4. Listen for male speakers using male reference voices and female speakers using female reference voices.
